In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from itertools import combinations
from rtree import index
import matplotlib.pyplot as plt
import seaborn as sns
import time

T1 = time.time()

# 指定分号为字段分隔符
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/4-2/03870.csv', sep=';') 
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/M3353_gpsdistance_normalized(2).csv') #20条
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/M3353_gpsdistance_normalized(2).csv') #10条
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/datas/20/M2503_gpsdistance_normalized.csv') #10条
fdata = pd.read_csv('E:/大数据/毕业设计/数据/datas/40/M2333_gpsdistance_normalized.csv') #10条
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/datas/10/M5583_gpsdistance_normalized.csv') #10条
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/4-2/M2333.csv', sep=';')  #40条
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/result/M4123_dropdata.csv')   #10条
display(fdata)

,idx,opath,lineName,direction,t_flag,time,lng,lat,distance_of_gpspoint,gps_normalized_distance_length
0,BS04317D,63083,M2333,1.0,1,2019-01-04 07:03:27,113.906742,22.709049,0.432755,0.010337
1,BS04317D,63083,M2333,1.0,1,2019-01-04 07:36:46,113.906392,22.709334,0.480602,0.011480
2,BS04317D,63083,M2333,1.0,1,2019-01-04 07:36:56,113.906379,22.709346,0.482525,0.011526
3,BS04317D,63083,M2333,1.0,1,2019-01-04 07:37:06,113.906344,22.709380,0.487659,0.011649
4,BS04317D,63083,M2333,1.0,1,2019-01-04 07:47:36,113.906416,22.709311,0.477055,0.011396
...,...,...,...,...,...,...,...,...,...,...
130962,BS08123D,75772,M2333,2.0,3,2019-01-04 16:05:15,113.904974,22.710708,41.662506,0.966983
130963,BS08123D,75771,M2333,2.0,3,2019-01-04 16:05:25,113.906310,22.709231,41.892125,0.972313
130964,BS08123D,75771,M2333,2.0,3,2019-01-04 16:05:35,113.906427,22.709129,41.908692,0.972697
130965,BS08123D,75771,M2333,2.0,3,2019-01-04 16:17:45,113.906521,22.709055,41.921378,0.972992


In [2]:
import pandas as pd

# 定义 assign_to_windows 函数
def assign_to_windows(row):
    current_time = row['time']
    minutes_since_midnight = (current_time - current_time.normalize()).total_seconds() / 60.0
    
    # 计算基本时间窗口的开始时间，确保每个窗口覆盖4分钟
    window_start_minute = int(minutes_since_midnight // 4 * 4)  # 使用整除并乘以4确保步进为4
    window_end_minute = window_start_minute + 4  # 窗口结束时间调整为开始时间后的4分钟
    
    return pd.Series([window_start_minute, window_end_minute])

# 将 'time' 列的数据类型从字符串转换为 datetime
fdata['time'] = pd.to_datetime(fdata['time'])

# 应用函数，并创建新的列来存储分配结果
fdata[['win_st', 'win_en']] = fdata.apply(assign_to_windows, axis=1)

In [3]:
# 基于车牌号（'idx'列）、线路号（'linenumber'列）和时间窗口的开始时间（'win_st'列）对数据进行分组
counts_per_group = fdata.groupby(['lineName', 'idx', 'win_st']).size()

# 过滤掉那些在同一时间窗口内每组只有一条记录的数据
# 这会返回一个包含每组在每个时间窗口内记录数大于1的过滤后的DataFrame的索引
filtered_idx = counts_per_group[counts_per_group > 1].index

# 基于过滤后的索引，我们选择原始数据集中符合条件的行
# 这里的'linenumber', 'idx'和'win_st'必须与groupby中使用的列名完全匹配
fdata_filtered = fdata[fdata.set_index(['lineName', 'idx', 'win_st']).index.isin(filtered_idx)]

display(fdata_filtered)

,idx,opath,lineName,direction,t_flag,time,lng,lat,distance_of_gpspoint,gps_normalized_distance_length,win_st,win_en
1,BS04317D,63083,M2333,1.0,1,2019-01-04 07:36:46,113.906392,22.709334,0.480602,0.011480,456,460
2,BS04317D,63083,M2333,1.0,1,2019-01-04 07:36:56,113.906379,22.709346,0.482525,0.011526,456,460
3,BS04317D,63083,M2333,1.0,1,2019-01-04 07:37:06,113.906344,22.709380,0.487659,0.011649,456,460
4,BS04317D,63083,M2333,1.0,1,2019-01-04 07:47:36,113.906416,22.709311,0.477055,0.011396,464,468
5,BS04317D,63083,M2333,1.0,1,2019-01-04 07:47:46,113.906424,22.709304,0.475932,0.011369,464,468
...,...,...,...,...,...,...,...,...,...,...,...,...
130962,BS08123D,75772,M2333,2.0,3,2019-01-04 16:05:15,113.904974,22.710708,41.662506,0.966983,964,968
130963,BS08123D,75771,M2333,2.0,3,2019-01-04 16:05:25,113.906310,22.709231,41.892125,0.972313,964,968
130964,BS08123D,75771,M2333,2.0,3,2019-01-04 16:05:35,113.906427,22.709129,41.908692,0.972697,964,968
130965,BS08123D,75771,M2333,2.0,3,2019-01-04 16:17:45,113.906521,22.709055,41.921378,0.972992,976,980


In [4]:
# 计算每辆车每个时间窗口的MBR
def calculate_mbrs(group):
    min_time = group['time'].min()
    max_time = group['time'].max()
    min_dis_location = group['gps_normalized_distance_length'].min()
    max_dis_location = group['gps_normalized_distance_length'].max()
    return pd.Series([min_time, max_time, min_dis_location, max_dis_location], index=['min_time', 'max_time', 'min_dis_location', 'max_dis_location'])

# 按 idx 和 win_st 分组，计算 MBR
mbrs = fdata_filtered.groupby(['idx', 'win_st']).apply(calculate_mbrs)
mbrs.reset_index(inplace=True)

# 将时间转换为时间戳
mbrs["min_timestamp"] = mbrs["min_time"].apply(lambda x: int(x.timestamp()))
mbrs["max_timestamp"] = mbrs["max_time"].apply(lambda x: int(x.timestamp()))

# 创建 R-Tree 索引
p = index.Property()
Mbr_Tree = index.Index(properties=p)

# 添加 MBR 到 R-Tree 索引
for mbr in mbrs.itertuples():
    Mbr_Tree.insert(mbr.Index, (mbr.min_timestamp, mbr.min_dis_location, mbr.max_timestamp, mbr.max_dis_location))

# 查找相交的 MBR
def find_intersections(mbrs, Mbr_Tree):
    intersection_pairs = []
    for mbr in mbrs.itertuples():
        intersecting_mbrs = list(Mbr_Tree.intersection((mbr.min_timestamp, mbr.min_dis_location, mbr.max_timestamp, mbr.max_dis_location), objects=True))
        for inter in intersecting_mbrs:
            if mbr.Index < inter.id:  # 防止重复对比
                intersection_pairs.append((mbr.Index, inter.id))
    return intersection_pairs

# 定义检查两个MBR是否相交的函数
def mbrs_intersect(mbr1, mbr2):
    return not (mbr1['max_timestamp'] < mbr2['min_timestamp'] or mbr1['min_timestamp'] > mbr2['max_timestamp'] or
                mbr1['max_dis_location'] < mbr2['min_dis_location'] or mbr1['min_dis_location'] > mbr2['max_dis_location'])

# 辅助函数，计算两点确定的直线的参数（斜率和截距）
def calculate_line_parameters(point1, point2):
    x_coords, y_coords = [point1[0], point2[0]], [point1[1], point2[1]]
    A = np.vstack([x_coords, np.ones(len(x_coords))]).T
    m, c = np.linalg.lstsq(A, y_coords, rcond=None)[0]
    return m, c

# 辅助函数，计算点到直线的距离
def point_to_line_distance(point, line_points):
    x0, y0 = point
    x1, y1 = line_points[0]
    x2, y2 = line_points[1]
    m, c = calculate_line_parameters((x1, y1), (x2, y2))
    A, B, C = -m, 1, -c
    distance = abs(A * x0 + B * y0 + C) / (np.sqrt(A**2 + B**2))
    return distance

# 函数，计算两条线段之间的最大垂直距离
def calculate_max_vertical_distance_between_lines(line1_points, line2_points):
    point1_line2, point2_line2, midpoint_line2 = line2_points
    distance_start = point_to_line_distance(point1_line2, line1_points)
    distance_mid = point_to_line_distance(midpoint_line2, line1_points)
    distance_end = point_to_line_distance(point2_line2, line1_points)
    return max(distance_start, distance_mid, distance_end)

# 计算串车事件
def detect_bus_bunching(data, intersection_pairs, threshold):
    bunching_events = []

    for (idx1, idx2) in intersection_pairs:
        mbr1 = mbrs.iloc[idx1]
        mbr2 = mbrs.iloc[idx2]

        if mbrs_intersect(mbr1, mbr2):
            line1_points = [(mbr1['min_timestamp'], mbr1['min_dis_location']), (mbr1['max_timestamp'], mbr1['max_dis_location'])]
            line2_points = [(mbr2['min_timestamp'], mbr2['min_dis_location']), (mbr2['max_timestamp'], mbr2['max_dis_location'])]
            midpoint_line2 = ((line2_points[0][0] + line2_points[1][0]) / 2, (line2_points[0][1] + line2_points[1][1]) / 2)
            distance = calculate_max_vertical_distance_between_lines(line1_points, [line2_points[0], midpoint_line2, line2_points[1]])
            
            T2 = time.time()
            print('程序运行时间:%s秒' % ((T2 - T1)*1))
            # 计算串车持续时间
            min_time = min(mbr1['min_time'], mbr2['min_time'])
            max_time = max(mbr1['max_time'], mbr2['max_time'])
            duration = max_time - min_time

            if distance < threshold:
                bunching_events.append({
                    'lineName': data.iloc[idx1]['lineName'],
                    'direction': data.iloc[idx1]['direction'],
                    'idx1': data.iloc[idx1]['idx'],
                    'idx2': data.iloc[idx2]['idx'],
                    'win_st': data.iloc[idx1]['win_st'],
                    'distance': distance,
                    'start_time': min_time,
                    'end_time': max_time,
                    'duration': duration
                })

    return pd.DataFrame(bunching_events)

# 查找相交的 MBR
intersection_pairs = find_intersections(mbrs, Mbr_Tree)

# 检测公交车串车
threshold = 0.01
bunching_df = detect_bus_bunching(fdata_filtered, intersection_pairs, threshold)

def merge_bunching_events(events):
    events = events.sort_values(by='win_st').reset_index(drop=True)
    merged_events = []
    i = 0

    while i < len(events):
        current_event = events.iloc[i]
        j = i + 1

        while j < len(events):
            next_event = events.iloc[j]
            if (current_event['idx1'] == next_event['idx1'] and current_event['idx2'] == next_event['idx2'] and
                (next_event['start_time'] - current_event['end_time']).total_seconds() < 5):
                current_event['end_time'] = next_event['end_time']
                current_event['duration'] = current_event['end_time'] - current_event['start_time']
                current_event['win_st'] = f"{current_event['win_st']},{next_event['win_st']}"
                j += 1
            else:
                break

        merged_events.append(current_event)
        i = j

    merged_df = pd.DataFrame(merged_events)
    # 过滤掉持续时间小于3分钟的事件
    filtered_merged_df = merged_df[merged_df['duration'] >= timedelta(minutes=3)]
    return filtered_merged_df

# 合并串车事件
merged_bunching_df = merge_bunching_events(bunching_df)

# 输出合并后的串车事件
print(merged_bunching_df)

程序运行时间:20.71381688117981秒
程序运行时间:20.71381688117981秒
程序运行时间:20.714824676513672秒
程序运行时间:20.714824676513672秒
程序运行时间:20.714824676513672秒
程序运行时间:20.716058254241943秒
程序运行时间:20.716058254241943秒
程序运行时间:20.716830492019653秒
程序运行时间:20.716830492019653秒
程序运行时间:20.717830657958984秒
程序运行时间:20.717830657958984秒
程序运行时间:20.717830657958984秒
程序运行时间:20.718831300735474秒
程序运行时间:20.718831300735474秒
程序运行时间:20.719831228256226秒
程序运行时间:20.719831228256226秒
程序运行时间:20.72083330154419秒
程序运行时间:20.72083330154419秒
程序运行时间:20.72083330154419秒
程序运行时间:20.72183132171631秒
程序运行时间:20.722831964492798秒
程序运行时间:20.722831964492798秒
程序运行时间:20.722831964492798秒
程序运行时间:20.7238552570343秒
程序运行时间:20.7238552570343秒
程序运行时间:20.724832773208618秒
程序运行时间:20.724832773208618秒
程序运行时间:20.72583246231079秒
程序运行时间:20.72583246231079秒
程序运行时间:20.7268328666687秒
程序运行时间:20.7268328666687秒
程序运行时间:20.7268328666687秒
程序运行时间:20.72807812690735秒
程序运行时间:20.72807812690735秒
程序运行时间:20.728833198547363秒
程序运行时间:20.728833198547363秒
程序运行时间:20.729820489883423秒
程序运行时间:20.72982048988

程序运行时间:21.32695460319519秒
程序运行时间:21.3319571018219秒
程序运行时间:21.33295726776123秒
程序运行时间:21.33295726776123秒
程序运行时间:21.333956003189087秒
程序运行时间:21.333956003189087秒
程序运行时间:21.334957361221313秒
程序运行时间:21.335958242416382秒
程序运行时间:21.336957931518555秒
程序运行时间:21.336957931518555秒
程序运行时间:21.337957859039307秒
程序运行时间:21.337957859039307秒
程序运行时间:21.33895707130432秒
程序运行时间:21.33895707130432秒
程序运行时间:21.339957237243652秒
程序运行时间:21.339957237243652秒
程序运行时间:21.340957403182983秒
程序运行时间:21.340957403182983秒
程序运行时间:21.34195828437805秒
程序运行时间:21.34195828437805秒
程序运行时间:21.34295892715454秒
程序运行时间:21.34295892715454秒
程序运行时间:21.34395956993103秒
程序运行时间:21.34395956993103秒
程序运行时间:21.34495973587036秒
程序运行时间:21.34596061706543秒
程序运行时间:21.34596061706543秒
程序运行时间:21.346960067749023秒
程序运行时间:21.347970724105835秒
程序运行时间:21.347970724105835秒
程序运行时间:21.348970890045166秒
程序运行时间:21.348970890045166秒
程序运行时间:21.34996747970581秒
程序运行时间:21.34996747970581秒
程序运行时间:21.34996747970581秒
程序运行时间:21.350966930389404秒
程序运行时间:21.350966930389404秒
程序运行时间:21.3509669303

程序运行时间:21.583014965057373秒
程序运行时间:21.584012746810913秒
程序运行时间:21.584012746810913秒
程序运行时间:21.585012197494507秒
程序运行时间:21.585012197494507秒
程序运行时间:21.586012363433838秒
程序运行时间:21.586012363433838秒
程序运行时间:21.587012767791748秒
程序运行时间:21.587012767791748秒
程序运行时间:21.5880126953125秒
程序运行时间:21.5880126953125秒
程序运行时间:21.58901286125183秒
程序运行时间:21.58901286125183秒
程序运行时间:21.59001326560974秒
程序运行时间:21.59001326560974秒
程序运行时间:21.59101366996765秒
程序运行时间:21.59101366996765秒
程序运行时间:21.592013835906982秒
程序运行时间:21.592013835906982秒
程序运行时间:21.593014001846313秒
程序运行时间:21.593014001846313秒
程序运行时间:21.594014167785645秒
程序运行时间:21.595014572143555秒
程序运行时间:21.595014572143555秒
程序运行时间:21.596014499664307秒
程序运行时间:21.596014499664307秒
程序运行时间:21.597014904022217秒
程序运行时间:21.597014904022217秒
程序运行时间:21.598015069961548秒
程序运行时间:21.598015069961548秒
程序运行时间:21.599015474319458秒
程序运行时间:21.599015474319458秒
程序运行时间:21.600016117095947秒
程序运行时间:21.600016117095947秒
程序运行时间:21.60101628303528秒
程序运行时间:21.60101628303528秒
程序运行时间:21.60201597213745秒
程序运行时间:21.6020

程序运行时间:22.28819751739502秒
程序运行时间:22.29019784927368秒
程序运行时间:22.29019784927368秒
程序运行时间:22.29119873046875秒
程序运行时间:22.29119873046875秒
程序运行时间:22.292198181152344秒
程序运行时间:22.292198181152344秒
程序运行时间:22.293198347091675秒
程序运行时间:22.293198347091675秒
程序运行时间:22.294198513031006秒
程序运行时间:22.295199155807495秒
程序运行时间:22.296199083328247秒
程序运行时间:22.296199083328247秒
程序运行时间:22.297199249267578秒
程序运行时间:22.297199249267578秒
程序运行时间:22.29819941520691秒
程序运行时间:22.29819941520691秒
程序运行时间:22.299200296401978秒
程序运行时间:22.299200296401978秒
程序运行时间:22.30020022392273秒
程序运行时间:22.30020022392273秒
程序运行时间:22.30120038986206秒
程序运行时间:22.30120038986206秒
程序运行时间:22.302200317382812秒
程序运行时间:22.302200317382812秒
程序运行时间:22.303200721740723秒
程序运行时间:22.303200721740723秒
程序运行时间:22.304200887680054秒
程序运行时间:22.304200887680054秒
程序运行时间:22.305201053619385秒
程序运行时间:22.305201053619385秒
程序运行时间:22.306201219558716秒
程序运行时间:22.306201219558716秒
程序运行时间:22.307202100753784秒
程序运行时间:22.307202100753784秒
程序运行时间:22.308201551437378秒
程序运行时间:22.308201551437378秒
程序运行时间:22.30

程序运行时间:22.809937000274658秒
程序运行时间:22.811939477920532秒
程序运行时间:22.811939477920532秒
程序运行时间:22.812939882278442秒
程序运行时间:22.812939882278442秒
程序运行时间:22.81394052505493秒
程序运行时间:22.81394052505493秒
程序运行时间:22.814940452575684秒
程序运行时间:22.814940452575684秒
程序运行时间:22.815940856933594秒
程序运行时间:22.816941022872925秒
程序运行时间:22.816941022872925秒
程序运行时间:22.817941188812256秒
程序运行时间:22.817941188812256秒
程序运行时间:22.818941354751587秒
程序运行时间:22.818941354751587秒
程序运行时间:22.819941759109497秒
程序运行时间:22.820415258407593秒
程序运行时间:22.820918798446655秒
程序运行时间:22.821919918060303秒
程序运行时间:22.821919918060303秒
程序运行时间:22.822920083999634秒
程序运行时间:22.822920083999634秒
程序运行时间:22.823920488357544秒
程序运行时间:22.823920488357544秒
程序运行时间:22.824920654296875秒
程序运行时间:22.82592248916626秒
程序运行时间:22.82592248916626秒
程序运行时间:22.826929092407227秒
程序运行时间:22.826929092407227秒
程序运行时间:22.826929092407227秒
程序运行时间:22.82792830467224秒
程序运行时间:22.82792830467224秒
程序运行时间:22.82892918586731秒
程序运行时间:22.82892918586731秒
程序运行时间:22.82993483543396秒
程序运行时间:22.82993483543396秒
程序运行时间:22.8

程序运行时间:23.643105506896973秒
程序运行时间:23.644105434417725秒
程序运行时间:23.64510464668274秒
程序运行时间:23.64510464668274秒
程序运行时间:23.64610505104065秒
程序运行时间:23.64610505104065秒
程序运行时间:23.64710521697998秒
程序运行时间:23.64710521697998秒
程序运行时间:23.64810538291931秒
程序运行时间:23.64810538291931秒
程序运行时间:23.64910578727722秒
程序运行时间:23.64910578727722秒
程序运行时间:23.650105714797974秒
程序运行时间:23.650105714797974秒
程序运行时间:23.651105880737305秒
程序运行时间:23.651105880737305秒
程序运行时间:23.652106285095215秒
程序运行时间:23.652106285095215秒
程序运行时间:23.653106689453125秒
程序运行时间:23.653106689453125秒
程序运行时间:23.654106855392456秒
程序运行时间:23.654106855392456秒
程序运行时间:23.655107021331787秒
程序运行时间:23.655107021331787秒
程序运行时间:23.656107187271118秒
程序运行时间:23.656107187271118秒
程序运行时间:23.65710735321045秒
程序运行时间:23.65710735321045秒
程序运行时间:23.65810751914978秒
程序运行时间:23.65810751914978秒
程序运行时间:23.659116506576538秒
程序运行时间:23.659116506576538秒
程序运行时间:23.660115242004395秒
程序运行时间:23.660115242004395秒
程序运行时间:23.660115242004395秒
程序运行时间:23.661121368408203秒
程序运行时间:23.661121368408203秒
程序运行时间:23.66212

    lineName  direction      idx1      idx2 win_st  distance  \
0      M2333        1.0  BS00047D  BS00047D    372  0.000744   
1      M2333        1.0  BS00027D  BS00027D    380  0.009197   
2      M2333        1.0  BS00027D  BS00027D    384  0.007580   
15     M2333        1.0  BS00027D  BS00047D    404  0.005349   
16     M2333        1.0  BS00027D  BS00027D    408  0.009226   
..       ...        ...       ...       ...    ...       ...   
480    M2333        2.0  BS04317D  BS00047D   1340  0.009441   
482    M2333        2.0  BS04317D  BS00027D   1356  0.007390   
486    M2333        2.0  BS04317D  BS00047D   1364  0.005367   
487    M2333        2.0  BS04317D  BS00047D   1368  0.006891   
488    M2333        2.0  BS04317D  BS00047D   1368  0.006269   

             start_time            end_time        duration  
0   2019-01-04 09:16:07 2019-01-04 09:19:55 0 days 00:03:48  
1   2019-01-04 16:16:06 2019-01-04 16:19:56 0 days 00:03:50  
2   2019-01-04 17:36:04 2019-01-04 17:39:56 0

C:\Users\ASUS\AppData\Local\Temp\ipykernel_21272\1082889789.py:121: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_event['end_time'] = next_event['end_time']
C:\Users\ASUS\AppData\Local\Temp\ipykernel_21272\1082889789.py:122: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_event['duration'] = current_event['end_time'] - current_event['start_time']
C:\Users\ASUS\AppData\Local\Temp\ipykernel_21272\1082889789.py:123: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.